# Feature Engineering — First 24-Hour Features

This notebook builds a model-ready feature DataFrame for the ICU mortality prediction task. For each of the 1,424 patients in the analytic cohort, we extract features from their first 24 hours of ICU stay.

## Feature categories

1. **Demographics** — age, gender, BMI (already in cohort table)
2. **Vital signs** — heart rate, blood pressure, respiratory rate, SpO2, temperature (aggregated min/max/mean over first 24 hours)
3. **Laboratory values** — creatinine, glucose, WBC, lactate, sodium, potassium, hemoglobin, bilirubin (aggregated over first 24 hours)
4. **Apache components** — Glasgow Coma Scale, admission diagnosis category

## Output

A DataFrame `features` with one row per patient, ~50 feature columns, and a binary target `in_hospital_mortality`. Saved to `data/features.parquet` (gitignored — do not commit).

## Setup: imports and database connection

In [1]:
import sys
sys.path.append("/Users/azadeh_sereshki/Documents/Projects/eicu-mortality-prediction/src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from db import get_engine

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11

engine = get_engine()
print("Connected. Ready to build features.")

Connected. Ready to build features.


## Pull demographics from the database

In [2]:
# Pull cohort + hospital characteristics into one query
# The cohort view already has age_numeric and all patient-level columns
# We join with the hospital table to get bed category, teaching status, region

demographics_query = """
    SELECT 
        c.patientunitstayid,
        c.patienthealthsystemstayid,
        c.age_numeric AS age,
        c.gender,
        c.admissionheight,
        c.admissionweight,
        c.unittype,
        c.hospitalid,
        h.numbedscategory,
        h.teachingstatus,
        h.region,
        c.hospitaldischargestatus
    FROM cohort c
    LEFT JOIN hospital h ON c.hospitalid = h.hospitalid
"""

demo = pd.read_sql(demographics_query, engine)
print(f"Shape: {demo.shape}")
print(f"Expected: (1424, 12)")
demo.head()

Shape: (1424, 12)
Expected: (1424, 12)


,patientunitstayid,patienthealthsystemstayid,age,gender,admissionheight,admissionweight,unittype,hospitalid,numbedscategory,teachingstatus,region,hospitaldischargestatus
0,147784,134042,60,Female,154.9,95.6,Med-Surg ICU,67,NaN,False,Midwest,Alive
1,151179,136669,59,Female,149.9,NaN,Med-Surg ICU,66,100 - 249,False,Midwest,Expired
2,151867,137216,44,Male,172.7,NaN,SICU,68,<100,False,Midwest,Alive
3,151900,137239,66,Female,165.1,86.8,MICU,73,>= 500,True,Midwest,Alive
4,152954,138053,41,Female,170.2,81.0,Med-Surg ICU,71,100 - 249,False,Midwest,Alive


## Compute BMI and the target variable

In [3]:
# Compute BMI from height (cm) and weight (kg)
# BMI = weight / (height/100)^2

demo["bmi"] = demo["admissionweight"] / ((demo["admissionheight"] / 100) ** 2)

# Create binary mortality target
# 1 = died in hospital, 0 = discharged alive
demo["in_hospital_mortality"] = (demo["hospitaldischargestatus"] == "Expired").astype(int)

# Quick validation
print("BMI summary statistics:")
print(demo["bmi"].describe())
print(f"\nMortality rate: {demo['in_hospital_mortality'].mean() * 100:.2f}%")
print(f"Deaths: {demo['in_hospital_mortality'].sum()} of {len(demo)}")

BMI summary statistics:
count    1356.000000
mean       30.221011
std        26.848874
min         2.291667
25%        23.369203
50%        27.526569
75%        32.793041
max       860.969388
Name: bmi, dtype: float64

Mortality rate: 8.29%
Deaths: 118 of 1424


## Check data quality on BMI

In [4]:
# Check for implausible BMI values
print("BMI value distribution:")
print(f"  BMI < 12 (implausibly low):  {(demo['bmi'] < 12).sum()} patients")
print(f"  BMI 12-60 (plausible):       {((demo['bmi'] >= 12) & (demo['bmi'] <= 60)).sum()} patients")
print(f"  BMI > 60 (implausibly high): {(demo['bmi'] > 60).sum()} patients")
print(f"  BMI missing:                 {demo['bmi'].isna().sum()} patients")

# Show extreme values for sanity-checking
print("\nMost extreme BMI values (likely data entry errors):")
print(demo.nlargest(5, "bmi")[["patientunitstayid", "admissionheight", "admissionweight", "bmi"]])
print()
print(demo.nsmallest(5, "bmi")[["patientunitstayid", "admissionheight", "admissionweight", "bmi"]])

BMI value distribution:
  BMI < 12 (implausibly low):  1 patients
  BMI 12-60 (plausible):       1342 patients
  BMI > 60 (implausibly high): 13 patients
  BMI missing:                 68 patients

Most extreme BMI values (likely data entry errors):
      patientunitstayid  admissionheight  admissionweight         bmi
1293            3135484             33.6             97.2  860.969388
1006            2684922             58.8            110.1  318.443704
1352            3157608             71.0            100.0  198.373339
471             1143992             67.9             90.0  195.210404
300              842498            167.6            515.0  183.340833

      patientunitstayid  admissionheight  admissionweight        bmi
226              482789            600.0             82.5   2.291667
372             1054428            167.0             35.0  12.549751
184              393867            187.9             44.7  12.660591
1218            3012414            170.2             

## Clean implausible values and finalize demographics

In [6]:
# Clean implausible BMI values
# Clinical plausibility range: 12 <= BMI <= 60
# Values outside this range are almost certainly data entry errors
demo["bmi_missing"] = demo["bmi"].isna() | (demo["bmi"] < 12) | (demo["bmi"] > 60)
demo.loc[(demo["bmi"] < 12) | (demo["bmi"] > 60), "bmi"] = np.nan

# Clean implausible height (should be 100-220cm for adults)
# Keep valid heights, null the rest
demo.loc[(demo["admissionheight"] < 100) | (demo["admissionheight"] > 220), "admissionheight"] = np.nan

# Clean implausible weight (should be 30-300kg for adults)
demo.loc[(demo["admissionweight"] < 30) | (demo["admissionweight"] > 300), "admissionweight"] = np.nan

# Encode gender as binary (1 = Male, 0 = Female, NaN for anything else)
demo["gender_male"] = (demo["gender"] == "Male").astype(int)

# Encode teaching status as 0/1 (keep NaN for missing)
demo["teaching_hospital"] = demo["teachingstatus"].astype(float)  # True→1.0, False→0.0, NaN→NaN

# Confirm the cleanup
print(f"BMI after cleanup:")
print(demo["bmi"].describe())
print(f"\nMissing BMI: {demo['bmi'].isna().sum()} patients (was {68}, now includes the 14 implausible cases)")
print(f"\nFeature preview:")

display_cols = ["patientunitstayid", "age", "gender_male", "bmi", "bmi_missing", "unittype", "numbedscategory", "teaching_hospital", "in_hospital_mortality"]
demo[display_cols].head()

BMI after cleanup:
count    1342.000000
mean       28.685017
std         7.713483
min        12.549751
25%        23.266711
50%        27.460454
75%        32.564220
max        58.906298
Name: bmi, dtype: float64

Missing BMI: 82 patients (was 68, now includes the 14 implausible cases)

Feature preview:


,patientunitstayid,age,gender_male,bmi,bmi_missing,unittype,numbedscategory,teaching_hospital,in_hospital_mortality
0,147784,60,0,39.843278,False,Med-Surg ICU,NaN,0.0,0
1,151179,59,0,NaN,True,Med-Surg ICU,100 - 249,0.0,1
2,151867,44,1,NaN,True,SICU,<100,0.0,0
3,151900,66,0,31.843851,False,MICU,>= 500,1.0,0
4,152954,41,0,27.961850,False,Med-Surg ICU,100 - 249,0.0,0


## Build the final demographics DataFrame

In [8]:
# Create the features DataFrame
# This is the base that we'll join vitals, labs, and Apache data onto

features = demo[[
    "patientunitstayid",        # primary key — used to join in more features later
    "age",
    "gender_male",
    "admissionheight",
    "admissionweight",
    "bmi",
    "bmi_missing",
    "unittype",
    "numbedscategory",
    "teaching_hospital",
    "region",
    "in_hospital_mortality",    # the target
]].copy()

print(f"Features DataFrame shape: {features.shape}")
print(f"\nColumn list:")

for col in features.columns:
    n_missing = features[col].isna().sum()
    dtype = features[col].dtype
    print(f"  {col:30s} {str(dtype):15s} missing: {n_missing}")

Features DataFrame shape: (1424, 12)

Column list:
  patientunitstayid              int64           missing: 0
  age                            int64           missing: 0
  gender_male                    int64           missing: 0
  admissionheight                float64         missing: 35
  admissionweight                float64         missing: 55
  bmi                            float64         missing: 82
  bmi_missing                    bool            missing: 0
  unittype                       str             missing: 0
  numbedscategory                str             missing: 211
  teaching_hospital              float64         missing: 0
  region                         str             missing: 127
  in_hospital_mortality          int64           missing: 0


## Check the raw teachingstatus values I pulled from the database

In [9]:
print("Raw teachingstatus value counts (from the demographics query):")
print(demo["teachingstatus"].value_counts(dropna=False))
print(f"\nNulls in teachingstatus: {demo['teachingstatus'].isna().sum()}")

# Check if teaching_hospital has the expected missing values
print(f"\nNulls in teaching_hospital (after astype(float)): {demo['teaching_hospital'].isna().sum()}")

# And check the same for region
print(f"\nRegion value counts:")
print(demo["region"].value_counts(dropna=False))

Raw teachingstatus value counts (from the demographics query):
teachingstatus
False    1277
True      147
Name: count, dtype: int64

Nulls in teachingstatus: 0

Nulls in teaching_hospital (after astype(float)): 0

Region value counts:
region
Midwest      465
South        454
West         284
NaN          127
Northeast     94
Name: count, dtype: int64


In [ ]:
import os
os.makedirs("../data", exist_ok=True)

features.to_parquet("../data/features_demographics.parquet", index=False)
print(f"Saved {len(features)} rows × {len(features.columns)} columns to data/features_demographics.parquet")

test_reload = pd.read_parquet("../data/features_demographics.parquet")
print(f"Reload verified: {test_reload.shape}")